# Step 0. DB 스키마 파악

- 목적: 테이블 목록, 컬럼·타입, 행 수, 날짜 범위, 기본키, 종목 코드 체계를 조사하고 이 과제에 쓸 수 있는 테이블을 구분한다.
- 규칙: `.env` 값은 출력하지 않는다. SELECT/SHOW만 실행한다.
- 검증 구간(2026-07-01~08-31)과 그 이후는 **건수·날짜 범위만** 확인하고 값은 보지 않는다.
- 결과 정리: `outputs/data_dictionary.md`

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.db import get_engine, read_sql

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 80)

TRAIN_START, TRAIN_END = "2024-01-01", "2026-06-30"
VALID_START, VALID_END = "2026-07-01", "2026-08-31"

engine = get_engine()
q = lambda sql, **kw: read_sql(sql, engine, params=kw or None)

## 1. 접속 확인과 권한

In [2]:
# DB 이름·계정명도 .env 값이므로 출력하지 않는다
display(q("SELECT VERSION() AS mysql_version"))
grants = q("SHOW GRANTS").iloc[:, 0]
grants = grants.str.replace(r"ON `[^`]+`\.", "ON <DB>.", regex=True).str.replace(r"TO .*$", "", regex=True)
grants.to_frame("grant")

,mysql_version
0,8.0.46


,grant
0,GRANT USAGE ON *.*
1,GRANT SELECT ON <DB>.*


## 2. 테이블 목록

In [3]:
tables = q("""
SELECT table_name AS tbl, table_type, engine, table_rows AS approx_rows,
       ROUND(data_length/1024/1024, 1) AS data_mb
FROM information_schema.tables WHERE table_schema = DATABASE() ORDER BY table_name""")
tables

,tbl,TABLE_TYPE,ENGINE,approx_rows,data_mb
0,collection_log,BASE TABLE,InnoDB,1725,0.1
1,kospi200_membership,BASE TABLE,InnoDB,6805,0.4
2,market_indicator_daily_candles,BASE TABLE,InnoDB,5274,1.5
3,market_indicator_investor_trading,BASE TABLE,InnoDB,1320,0.2
4,market_institution_breakdown,BASE TABLE,InnoDB,9240,1.5
5,stock_credit_trades,BASE TABLE,InnoDB,136954,22.6
6,stock_daily_candles,BASE TABLE,InnoDB,132379,20.5
7,stock_institution_breakdown,BASE TABLE,InnoDB,1046601,117.7
8,stock_investor_trading,BASE TABLE,InnoDB,145664,39.6
9,stock_minute_candles,BASE TABLE,InnoDB,0,0.0


## 3. 컬럼·타입·키

In [4]:
cols = q("""
SELECT table_name AS tbl, ordinal_position AS pos, column_name AS col, column_type AS type,
       is_nullable AS nullable, column_key AS `key`, column_comment AS comment
FROM information_schema.columns WHERE table_schema = DATABASE()
ORDER BY table_name, ordinal_position""")
for t, g in cols.groupby("tbl"):
    print(f"=== {t}")
    print(g.drop(columns="tbl").to_string(index=False), "\n")

=== collection_log
 pos             col                     type nullable key comment
   1              id                   bigint       NO PRI        
   2  run_started_at                 datetime       NO            
   3 run_finished_at                 datetime      YES            
   4            task              varchar(50)       NO MUL        
   5          symbol              varchar(20)      YES            
   6          status enum('SUCCESS','FAILED')       NO            
   7   rows_affected                      int      YES            
   8   error_message                     text      YES             

=== kospi200_membership
 pos           col        type nullable key comment
   1 snapshot_date        date       NO PRI        
   2        symbol varchar(20)       NO PRI        
   3  collected_at    datetime       NO             

=== market_indicator_daily_candles
 pos          col          type nullable key comment
   1       symbol   varchar(20)       NO PRI        
 

In [5]:
idx = q("""
SELECT table_name AS tbl, index_name, GROUP_CONCAT(column_name ORDER BY seq_in_index) AS cols,
       MAX(non_unique) AS non_unique
FROM information_schema.statistics WHERE table_schema = DATABASE()
GROUP BY table_name, index_name ORDER BY table_name, index_name""")
idx

,tbl,INDEX_NAME,cols,non_unique
0,collection_log,idx_collection_log_task,"task,run_started_at",1
1,collection_log,PRIMARY,id,0
2,kospi200_membership,idx_kospi200_membership_symbol,symbol,1
3,kospi200_membership,PRIMARY,"snapshot_date,symbol",0
4,market_indicator_daily_candles,PRIMARY,"symbol,trade_date",0
5,market_indicator_investor_trading,PRIMARY,"symbol,trade_date",0
6,market_institution_breakdown,PRIMARY,"symbol,trade_date,category",0
7,stock_credit_trades,idx_credit_trades_date,trade_date,1
8,stock_credit_trades,PRIMARY,"symbol,trade_date",0
9,stock_daily_candles,idx_daily_candles_date,trade_date,1


## 4. 정확한 행 수, 날짜 범위, 종목 수

In [6]:
DATE_COL = {
    "kospi200_membership": "snapshot_date",
    "market_indicator_daily_candles": "trade_date",
    "market_indicator_investor_trading": "trade_date",
    "market_institution_breakdown": "trade_date",
    "stock_credit_trades": "trade_date",
    "stock_daily_candles": "trade_date",
    "stock_institution_breakdown": "trade_date",
    "stock_investor_trading": "trade_date",
    "stock_minute_candles": "ts",
    "stock_program_trades": "trade_date",
    "stock_securities_lending": "trade_date",
    "stock_short_selling": "trade_date",
    "stock_warnings": "start_date",
}
rows = []
for t, d in DATE_COL.items():
    r = q(f"""
    SELECT COUNT(*) AS n_rows, COUNT(DISTINCT {d}) AS n_dates, MIN({d}) AS date_min, MAX({d}) AS date_max,
           COUNT(DISTINCT symbol) AS n_symbols,
           SUM({d} < :ts) AS n_pre2024,
           SUM({d} BETWEEN :ts AND :te) AS n_train,
           SUM({d} BETWEEN :vs AND :ve) AS n_valid,
           SUM({d} > :ve) AS n_after_valid
    FROM {t}""", ts=TRAIN_START, te=TRAIN_END, vs=VALID_START, ve=VALID_END)
    r.insert(0, "tbl", t)
    rows.append(r)
summary = pd.concat(rows, ignore_index=True)
extra = q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT symbol) AS n_symbols FROM stocks").assign(tbl="stocks")
summary = pd.concat([summary, extra], ignore_index=True)
summary

,tbl,n_rows,n_dates,date_min,date_max,n_symbols,n_pre2024,n_train,n_valid,n_after_valid
0,kospi200_membership,6805,34.0,2024-01-01,2026-09-16,232,0.0,6003.0,400.0,402.0
1,market_indicator_daily_candles,5274,660.0,2024-01-02,2026-09-16,8,0.0,4848.0,336.0,90.0
2,market_indicator_investor_trading,1320,660.0,2024-01-02,2026-09-16,2,0.0,1212.0,84.0,24.0
3,market_institution_breakdown,9240,660.0,2024-01-02,2026-09-16,2,0.0,8484.0,588.0,168.0
4,stock_credit_trades,149869,661.0,2024-01-02,2026-09-15,230,0.0,137716.0,9623.0,2530.0
5,stock_daily_candles,149652,660.0,2024-01-02,2026-09-16,230,0.0,137269.0,9623.0,2760.0
6,stock_institution_breakdown,1047557,660.0,2024-01-02,2026-09-16,230,0.0,960883.0,67361.0,19313.0
7,stock_investor_trading,149652,660.0,2024-01-02,2026-09-16,230,0.0,137269.0,9623.0,2760.0
8,stock_minute_candles,0,0.0,None,None,0,NaN,NaN,NaN,NaN
9,stock_program_trades,149133,660.0,2024-01-02,2026-09-16,230,0.0,136822.0,9563.0,2748.0


In [7]:
# 구간별 거래일 수 (stock_daily_candles 기준)
q("""
SELECT SUM(d BETWEEN :ts AND :te) AS train_days, SUM(d BETWEEN :vs AND :ve) AS valid_days,
       SUM(d > :ve) AS after_valid_days
FROM (SELECT DISTINCT trade_date AS d FROM stock_daily_candles) x""",
  ts=TRAIN_START, te=TRAIN_END, vs=VALID_START, ve=VALID_END)

,train_days,valid_days,after_valid_days
0,606.0,42.0,12.0


## 5. 종목 코드 체계

In [8]:
code_check = []
for t in list(DATE_COL) + ["stocks"]:
    r = q(f"""
    SELECT COUNT(DISTINCT symbol) AS n_symbols,
           SUM(symbol REGEXP '^[0-9A-Z]{{6}}$') AS n_rows_6char,
           SUM(symbol REGEXP '^[0-9]{{6}}$') AS n_rows_6digit,
           COUNT(*) AS n_rows
    FROM {t}""")
    r.insert(0, "tbl", t)
    code_check.append(r)
pd.concat(code_check, ignore_index=True)

,tbl,n_symbols,n_rows_6char,n_rows_6digit,n_rows
0,kospi200_membership,232,6805.0,6792.0,6805
1,market_indicator_daily_candles,8,660.0,0.0,5274
2,market_indicator_investor_trading,2,660.0,0.0,1320
3,market_institution_breakdown,2,4620.0,0.0,9240
4,stock_credit_trades,230,149869.0,149653.0,149869
5,stock_daily_candles,230,149652.0,149435.0,149652
6,stock_institution_breakdown,230,1047557.0,1046038.0,1047557
7,stock_investor_trading,230,149652.0,149435.0,149652
8,stock_minute_candles,0,NaN,NaN,0
9,stock_program_trades,230,149133.0,148918.0,149133


In [9]:
# 지수·금리 테이블의 심볼 (일반 종목 코드와 체계가 다름)
q("""
SELECT 'market_indicator_daily_candles' AS tbl, symbol, COUNT(*) AS n, MIN(trade_date) AS d0, MAX(trade_date) AS d1
FROM market_indicator_daily_candles GROUP BY symbol
UNION ALL
SELECT 'market_indicator_investor_trading', symbol, COUNT(*), MIN(trade_date), MAX(trade_date)
FROM market_indicator_investor_trading GROUP BY symbol
UNION ALL
SELECT 'market_institution_breakdown', symbol, COUNT(*), MIN(trade_date), MAX(trade_date)
FROM market_institution_breakdown GROUP BY symbol""")

,tbl,symbol,n,d0,d1
0,market_indicator_daily_candles,KOSDAQ,660,2024-01-02,2026-09-16
1,market_indicator_daily_candles,KOSPI,660,2024-01-02,2026-09-16
2,market_indicator_daily_candles,KR_BOND_10Y,659,2024-01-02,2026-09-15
3,market_indicator_daily_candles,KR_BOND_20Y,659,2024-01-02,2026-09-15
4,market_indicator_daily_candles,KR_BOND_2Y,659,2024-01-02,2026-09-15
5,market_indicator_daily_candles,KR_BOND_30Y,659,2024-01-02,2026-09-15
6,market_indicator_daily_candles,KR_BOND_3Y,659,2024-01-02,2026-09-15
7,market_indicator_daily_candles,KR_BOND_5Y,659,2024-01-02,2026-09-15
8,market_indicator_investor_trading,KOSDAQ,660,2024-01-02,2026-09-16
9,market_indicator_investor_trading,KOSPI,660,2024-01-02,2026-09-16


## 6. 종목 마스터(`stocks`)

In [10]:
q("""
SELECT market, security_type, is_common_share, status, currency, is_pilot,
       COUNT(*) AS n, SUM(delist_date IS NOT NULL) AS n_delist_date,
       SUM(industry IS NULL) AS n_industry_null, COUNT(DISTINCT industry) AS n_industry,
       MIN(list_date) AS list_min, MAX(list_date) AS list_max, MAX(updated_at) AS updated_max
FROM stocks GROUP BY 1,2,3,4,5,6""")

,market,security_type,is_common_share,status,currency,is_pilot,n,n_delist_date,n_industry_null,n_industry,list_min,list_max,updated_max
0,KOSDAQ,STOCK,1,ACTIVE,KRW,1,1,0.0,0.0,1,1994-11-07,1994-11-07,2026-09-17 13:01:44
1,KOSPI,STOCK,1,ACTIVE,KRW,1,229,0.0,1.0,68,1956-07-02,2026-08-25,2026-09-17 13:01:44


## 7. KOSPI200 구성종목 테이블

In [11]:
snap = q("""
SELECT snapshot_date, COUNT(*) AS n_members, MIN(collected_at) AS collected_min, MAX(collected_at) AS collected_max
FROM kospi200_membership GROUP BY snapshot_date ORDER BY snapshot_date""")
snap

,snapshot_date,n_members,collected_min,collected_max
0,2024-01-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
1,2024-02-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
2,2024-03-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
3,2024-04-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
4,2024-05-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
5,2024-06-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
6,2024-07-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
7,2024-08-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
8,2024-09-01,200,2026-09-16 18:17:40,2026-09-16 18:17:40
9,2024-10-01,201,2026-09-16 18:17:40,2026-09-16 18:17:40


In [12]:
# 구성종목 이력에는 있으나 종목 마스터·일봉에 없는 종목
q("""
SELECT m.symbol, COUNT(*) AS n_snapshots, MIN(m.snapshot_date) AS first_snap, MAX(m.snapshot_date) AS last_snap,
       (SELECT COUNT(*) FROM stocks s WHERE s.symbol = m.symbol) AS in_stocks,
       (SELECT COUNT(*) FROM stock_daily_candles c WHERE c.symbol = m.symbol) AS n_candles
FROM kospi200_membership m
WHERE m.symbol NOT IN (SELECT symbol FROM stocks)
GROUP BY m.symbol""")

,symbol,n_snapshots,first_snap,last_snap,in_stocks,n_candles
0,003410,6,2024-01-01,2024-06-01,0,0
1,010620,23,2024-01-01,2025-11-01,0,0
2,042670,24,2024-01-01,2025-12-01,0,0


In [13]:
# 종목 마스터에는 있으나 구성종목 이력에 한 번도 없는 종목
q("""SELECT symbol, name, market FROM stocks
WHERE symbol NOT IN (SELECT DISTINCT symbol FROM kospi200_membership)""")

,symbol,name,market
0,010170,대한광통신,KOSDAQ


In [14]:
# 누락 종목의 수집 실패 로그
q("""
SELECT task, symbol, status, LEFT(error_message, 60) AS error
FROM collection_log WHERE status = 'FAILED' ORDER BY symbol, task""")

,task,symbol,status,error
0,candles,003410,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gXShnPK846o
1,credit_trades,003410,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gfTbiuUE5v6
2,investor_trading,003410,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gxscS3XVei5
3,program_trades,003410,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=g4ZY4k1FsCq
4,securities_lending,003410,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gSErJDNmw1s
5,short_selling,003410,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gcNThvSVzv3
6,warnings,003410,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gtZY4BR4XNn
7,candles,010620,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gihOs6YdtaS
8,credit_trades,010620,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gzV7xIL4RSP
9,investor_trading,010620,FAILED,[404] stock-not-found: 종목을 찾을 수 없습니다. (requestId=gp7lazGU6Of


In [15]:
# stock_daily_candles.is_kospi200 이 '해당일 이전 가장 최근 월초 스냅샷'과 일치하는지 (학습 구간)
q("""
SELECT COUNT(*) AS n,
       SUM(c.is_kospi200 = (m.symbol IS NOT NULL)) AS n_match,
       SUM(c.is_kospi200 <> (m.symbol IS NOT NULL)) AS n_mismatch,
       SUM(c.is_kospi200 IS NULL) AS n_null
FROM stock_daily_candles c
JOIN (SELECT d.trade_date, MAX(s.snapshot_date) AS sd
      FROM (SELECT DISTINCT trade_date FROM stock_daily_candles WHERE trade_date BETWEEN :ts AND :te) d
      JOIN (SELECT DISTINCT snapshot_date FROM kospi200_membership) s ON s.snapshot_date <= d.trade_date
      GROUP BY d.trade_date) k USING (trade_date)
LEFT JOIN kospi200_membership m ON m.snapshot_date = k.sd AND m.symbol = c.symbol""",
  ts=TRAIN_START, te=TRAIN_END)

,n,n_match,n_mismatch,n_null
0,137269,137269.0,0.0,0.0


In [16]:
# 학습 구간 일자별 is_kospi200=1 종목 수 분포
q("""
SELECT n_in, COUNT(*) AS n_days, MIN(trade_date) AS d0, MAX(trade_date) AS d1
FROM (SELECT trade_date, SUM(is_kospi200 = 1) AS n_in FROM stock_daily_candles
      WHERE trade_date BETWEEN :ts AND :te GROUP BY trade_date) x
GROUP BY n_in ORDER BY d0""", ts=TRAIN_START, te=TRAIN_END)

,n_in,n_days,d0,d1
0,197.0,121,2024-01-02,2024-06-28
1,198.0,283,2024-07-01,2025-11-28
2,199.0,82,2024-10-02,2025-12-30
3,200.0,120,2026-01-02,2026-06-30


## 8. 테이블 간 거래일 캘린더 비교

In [17]:
cal = []
for t in ["stock_credit_trades", "stock_securities_lending", "stock_short_selling", "stock_investor_trading",
          "stock_program_trades", "stock_institution_breakdown", "market_indicator_daily_candles"]:
    r = q(f"""
    SELECT '{t}' AS tbl,
      (SELECT GROUP_CONCAT(DISTINCT trade_date) FROM {t}
        WHERE trade_date NOT IN (SELECT DISTINCT trade_date FROM stock_daily_candles)) AS extra_dates,
      (SELECT COUNT(DISTINCT trade_date) FROM stock_daily_candles
        WHERE trade_date NOT IN (SELECT DISTINCT trade_date FROM {t})) AS n_missing_dates""")
    cal.append(r)
pd.concat(cal, ignore_index=True)

,tbl,extra_dates,n_missing_dates
0,stock_credit_trades,"2024-12-31,2025-12-31",1
1,stock_securities_lending,"2024-12-31,2025-12-31",1
2,stock_short_selling,None,0
3,stock_investor_trading,None,0
4,stock_program_trades,None,0
5,stock_institution_breakdown,None,0
6,market_indicator_daily_candles,None,0


In [18]:
# stock_institution_breakdown: (종목, 일자)별 기관 세부 분류 개수
q("""SELECT n_cat, COUNT(*) AS n FROM
(SELECT symbol, trade_date, COUNT(*) AS n_cat FROM stock_institution_breakdown GROUP BY 1, 2) x GROUP BY n_cat""")

,n_cat,n
0,7,149651


## 9. 공표 시차 추정 (`updated_at` - `trade_date`, 학습 구간)

`updated_at`은 원천 데이터의 최종 갱신 시각으로 보인다. 최초 공표 시각과 같다고 단정할 수 없으므로 **참고용**이다.
장 마감(15:30) 이후 갱신되는 값은 엄격한 "t일 장 마감 시점" 기준에서는 t일 변수로 쓰기 애매하다.

In [19]:
lag = []
for t in ["stock_investor_trading", "stock_credit_trades", "stock_securities_lending",
          "stock_short_selling", "market_indicator_investor_trading"]:
    r = q(f"""
    SELECT '{t}' AS tbl, DATEDIFF(DATE(updated_at), trade_date) AS lag_days, COUNT(*) AS n,
           MIN(TIME(updated_at)) AS time_min, MAX(TIME(updated_at)) AS time_max
    FROM {t} WHERE trade_date BETWEEN :ts AND :te GROUP BY 2""", ts=TRAIN_START, te=TRAIN_END)
    r["share"] = (r["n"] / r["n"].sum()).round(3)
    lag.append(r.sort_values("n", ascending=False).head(4))
pd.concat(lag, ignore_index=True)

,tbl,lag_days,n,time_min,time_max,share
0,stock_investor_trading,1,103566,0 days 06:50:01,0 days 18:22:50,0.754
1,stock_investor_trading,3,26730,0 days 06:50:01,0 days 12:32:39,0.195
2,stock_investor_trading,2,3221,0 days 06:50:01,0 days 09:41:35,0.023
3,stock_investor_trading,4,1816,0 days 06:50:01,0 days 06:52:39,0.013
4,stock_credit_trades,1,104421,0 days 04:00:30,0 days 14:18:32,0.758
5,stock_credit_trades,3,26729,0 days 04:00:31,0 days 07:38:07,0.194
6,stock_credit_trades,2,3165,0 days 04:00:31,0 days 04:00:54,0.023
7,stock_credit_trades,4,1816,0 days 04:00:33,0 days 04:00:51,0.013
8,stock_securities_lending,0,79648,0 days 19:00:34,0 days 21:14:49,0.578
9,stock_securities_lending,2,20702,0 days 19:00:32,0 days 19:01:06,0.150


## 10. NULL 허용 컬럼의 결측 건수 (학습 구간, 개요만 — 상세는 Step 2)

In [20]:
nullable = cols[(cols["nullable"] == "YES") & cols["tbl"].isin(list(DATE_COL))]
out = []
for t, g in nullable.groupby("tbl"):
    d = DATE_COL[t]
    expr = ", ".join(f"SUM(`{c}` IS NULL) AS `{c}`" for c in g["col"])
    r = q(f"SELECT COUNT(*) AS n_rows, {expr} FROM {t} WHERE {d} BETWEEN :ts AND :te", ts=TRAIN_START, te=TRAIN_END)
    r = r.melt(id_vars="n_rows", var_name="col", value_name="n_null")
    r.insert(0, "tbl", t)
    out.append(r)
nulls = pd.concat(out, ignore_index=True)
nulls[nulls["n_null"] > 0]

,tbl,n_rows,col,n_null
20,stock_investor_trading,137269,cfd_buy_balance_qty,24554.0
21,stock_investor_trading,137269,cfd_buy_balance_rate,24554.0
22,stock_investor_trading,137269,cfd_sell_balance_qty,24554.0
23,stock_investor_trading,137269,cfd_sell_balance_rate,24554.0
24,stock_short_selling,133754,short_volume_rate,2158.0
25,stock_short_selling,133754,short_amount_rate,2158.0


## 11. 예측 시점(t+1일 09:00) 전에 확정되는지 점검

예측은 t+1일 장 시작 전에 한다. t일 장 마감 후 공표 데이터는 lag 없이 쓰므로, 원천 갱신 시각 `updated_at`이 **다음 거래일 09:00 이전**인지 확인한다.
`collected_at`(DB 적재 시각)은 2026-09-16~17에 소급 일괄 적재된 값이라 판단 근거로 쓸 수 없다.

In [21]:
# DB 적재 시각: 모두 소급 일괄 적재
pd.concat([q(f"SELECT '{t}' AS tbl, MIN(collected_at) AS c_min, MAX(collected_at) AS c_max FROM {t}")
           for t in ["stock_daily_candles", "stock_investor_trading", "stock_credit_trades", "stock_securities_lending",
                     "stock_short_selling", "stock_program_trades", "market_indicator_investor_trading"]],
          ignore_index=True)

,tbl,c_min,c_max
0,stock_daily_candles,2026-09-17 10:50:28,2026-09-17 10:50:28
1,stock_investor_trading,2026-09-16 14:33:54,2026-09-16 18:25:28
2,stock_credit_trades,2026-09-16 14:33:56,2026-09-16 18:29:13
3,stock_securities_lending,2026-09-16 14:34:00,2026-09-16 18:41:43
4,stock_short_selling,2026-09-16 14:33:59,2026-09-16 18:37:17
5,stock_program_trades,2026-09-16 14:33:57,2026-09-16 18:33:16
6,market_indicator_investor_trading,2026-09-16 14:34:03,2026-09-16 18:42:47


In [22]:
cal = pd.to_datetime(q("SELECT DISTINCT trade_date FROM stock_daily_candles ORDER BY 1")["trade_date"])
next_day = pd.Series(cal.shift(-1).values, index=cal.values)

late_rows = {}
res = []
for t in ["stock_investor_trading", "stock_credit_trades", "stock_securities_lending",
          "stock_short_selling", "market_indicator_investor_trading"]:
    d = q(f"SELECT symbol, trade_date, updated_at FROM {t} WHERE trade_date BETWEEN :ts AND :te",
          ts=TRAIN_START, te=TRAIN_END)
    d["trade_date"] = pd.to_datetime(d["trade_date"])
    d["deadline"] = d["trade_date"].map(next_day) + pd.Timedelta(hours=9)
    late = d["updated_at"] > d["deadline"]
    late_rows[t] = d[late]
    res.append({"tbl": t, "n_rows": len(d), "n_late": int(late.sum()), "late_share": round(late.mean(), 4),
                "n_late_dates": d.loc[late, "trade_date"].nunique()})
pd.DataFrame(res)

,tbl,n_rows,n_late,late_share,n_late_dates
0,stock_investor_trading,137269,693,0.0050,7
1,stock_credit_trades,137716,229,0.0017,2
2,stock_securities_lending,137712,32129,0.2333,144
3,stock_short_selling,133754,0,0.0000,0
4,market_indicator_investor_trading,1212,0,0.0000,0


In [23]:
# 늦게 갱신된 행: 기준일(trade_date)과 갱신일별 건수
for t, L in late_rows.items():
    if L.empty:
        continue
    g = (L.assign(updated_date=L["updated_at"].dt.date)
          .groupby("updated_date").agg(n=("symbol", "size"), td_min=("trade_date", "min"),
                                       td_max=("trade_date", "max"), n_dates=("trade_date", "nunique")))
    print(f"=== {t}")
    print(g.sort_values("n", ascending=False).to_string(), "\n")

=== stock_investor_trading
                n     td_min     td_max  n_dates
updated_date                                    
2024-01-15    396 2024-01-02 2024-01-03        2
2025-08-27    183 2025-08-19 2025-08-19        1
2026-04-22     56 2026-04-21 2026-04-21        1
2026-04-18     56 2026-04-16 2026-04-16        1
2026-01-19      1 2026-01-16 2026-01-16        1
2026-01-16      1 2026-01-15 2026-01-15        1 

=== stock_credit_trades
                n     td_min     td_max  n_dates
updated_date                                    
2025-08-27    228 2025-08-19 2025-08-19        1
2025-12-12      1 2025-12-11 2025-12-11        1 

=== stock_securities_lending
                  n     td_min     td_max  n_dates
updated_date                                      
2024-08-08    32129 2024-01-02 2024-07-31      144 

